<a href="https://colab.research.google.com/github/crystalclcm/JobPostings/blob/main/Post_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd

In [2]:
APP_ID = "90fb93ab"
APP_KEY = "ac0d74800544bfc63431a2d3747a92c5"


In [ ]:
import requests
import pandas as pd
import time
import random

# 🔐 Insert your keys here (keep private!)
APP_ID = "90fb93ab"
APP_KEY = "ac0d74800544bfc63431a2d3747a92c5"

BASE_URL = "https://api.adzuna.com/v1/api/jobs/{country}/search/{page}"

roles = [
    "Accountant",
    "Financial Analyst",
    "Business Analyst",
    "Risk Analyst",
    "Software Engineer",
    "Data Analyst",
    "Data Engineer",
    "Cybersecurity Analyst"
]

countries = ["gb", "de", "nl", "us", "ca", "au"]  # UK, Germany, Netherlands, US, Canada, Australia

def fetch_adzuna_jobs(country, role, pages=10, results_per_page=50):
    all_results = []

    for page in range(1, pages + 1):
        print(f"Country: {country.upper()} | Role: {role} | Page: {page}")

        params = {
            "app_id": APP_ID,
            "app_key": APP_KEY,
            "results_per_page": results_per_page,
            "what": role,
            "content-type": "application/json"
        }

        url = BASE_URL.format(country=country, page=page)

        try:
            r = requests.get(url, params=params, timeout=15)
            if r.status_code != 200:
                print(f"  ⚠️ Status code {r.status_code} for {country} - {role} - page {page}")
                continue

            data = r.json()
            jobs = data.get("results", [])

            for job in jobs:
                all_results.append({
                    "role": role,
                    "country": country.upper(),
                    "job_title": job.get("title"),
                    "company": job.get("company", {}).get("display_name"),
                    "location": job.get("location", {}).get("display_name"),
                    "created": job.get("created"),
                    "redirect_url": job.get("redirect_url"),
                    "description": job.get("description")
                })

        except Exception as e:
            print(f"  ❌ Error for {country} - {role} - page {page}: {e}")

        time.sleep(random.uniform(1.0, 2.5))  # polite delay

    return all_results

all_results = []

for country in countries:
    for role in roles:
        results = fetch_adzuna_jobs(country, role, pages=10, results_per_page=50)
        all_results.extend(results)

df = pd.DataFrame(all_results)
print("Total rows scraped:", len(df))

df.to_csv("adzuna_expanded_combined.csv", index=False)
df.head()


Country: GB | Role: Accountant | Page: 1
Country: GB | Role: Accountant | Page: 2
Country: GB | Role: Accountant | Page: 3
Country: GB | Role: Accountant | Page: 4


In [3]:
import os
os.makedirs("data/post_ai/adzuna", exist_ok=True)


In [4]:
import requests
import pandas as pd

APP_ID = "90fb93ab"
APP_KEY = "ac0d74800544bfc63431a2d3747a92c5"

def get_adzuna_jobs(country_code, role):
    url = f"https://api.adzuna.com/v1/api/jobs/{country_code}/search/1"
    params = {
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "results_per_page": 50,
        "what": role
    }

    response = requests.get(url, params=params).json()
    results = response.get("results", [])

    rows = []
    for job in results:
        rows.append({
            "job_title": job.get("title"),
            "job_description": job.get("description"),
            "country": country_code,
            "date_posted": job.get("created"),
            "source": "adzuna"
        })

    return pd.DataFrame(rows)


In [5]:
df = get_adzuna_jobs("gb", "accountant")
df.to_csv("data/post_ai/adzuna/accountant_gb.csv", index=False)

df = get_adzuna_jobs("de", "accountant")
df.to_csv("data/post_ai/adzuna/accountant_de.csv", index=False)

df = get_adzuna_jobs("nl", "accountant")
df.to_csv("data/post_ai/adzuna/accountant_nl.csv", index=False)


In [6]:
df = get_adzuna_jobs("gb", "business analyst")
df.to_csv("data/post_ai/adzuna/business_analyst_gb.csv", index=False)

df = get_adzuna_jobs("de", "business analyst")
df.to_csv("data/post_ai/adzuna/business_analyst_de.csv", index=False)

df = get_adzuna_jobs("nl", "business analyst")
df.to_csv("data/post_ai/adzuna/business_analyst_nl.csv", index=False)


In [7]:
df = get_adzuna_jobs("gb", "data analyst")
df.to_csv("data/post_ai/adzuna/data_analyst_gb.csv", index=False)

df = get_adzuna_jobs("de", "data analyst")
df.to_csv("data/post_ai/adzuna/data_analyst_de.csv", index=False)

df = get_adzuna_jobs("nl", "data analyst")
df.to_csv("data/post_ai/adzuna/data_analyst_nl.csv", index=False)


In [8]:
df = get_adzuna_jobs("gb", "software engineer")
df.to_csv("data/post_ai/adzuna/software_engineer_gb.csv", index=False)

df = get_adzuna_jobs("de", "software engineer")
df.to_csv("data/post_ai/adzuna/software_engineer_de.csv", index=False)

df = get_adzuna_jobs("nl", "software engineer")
df.to_csv("data/post_ai/adzuna/software_engineer_nl.csv", index=False)


# cleaning

In [9]:
import pandas as pd

def clean_adzuna(df):
    df = df.rename(columns={
        "title": "job_title",
        "job_title": "job_title",
        "description": "job_description",
        "job_description": "job_description",
        "created": "date_posted",
        "country": "country"
    })

    df["source"] = "adzuna"

    keep = ["job_title", "job_description", "country", "date_posted", "source"]
    df = df[[c for c in keep if c in df.columns]]

    return df


In [10]:
df = pd.read_csv("data/post_ai/adzuna/accountant_gb.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_accountant_gb.csv", index=False)


In [11]:
df = pd.read_csv("data/post_ai/adzuna/accountant_de.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_accountant_de.csv", index=False)


In [12]:
df = pd.read_csv("data/post_ai/adzuna/accountant_nl.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_accountant_nl.csv", index=False)


In [13]:
df = pd.read_csv("data/post_ai/adzuna/business_analyst_gb.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_business_analyst_gb.csv", index=False)


In [14]:
df = pd.read_csv("data/post_ai/adzuna/business_analyst_de.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_business_analyst_de.csv", index=False)


In [15]:
df = pd.read_csv("data/post_ai/adzuna/business_analyst_nl.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_business_analyst_nl.csv", index=False)

In [16]:
df = pd.read_csv("data/post_ai/adzuna/data_analyst_gb.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_data_analyst_gb.csv", index=False)


In [17]:
df = pd.read_csv("data/post_ai/adzuna/data_analyst_de.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_data_analyst_de.csv", index=False)


In [18]:
df = pd.read_csv("data/post_ai/adzuna/data_analyst_nl.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_data_analyst_nl.csv", index=False)


In [19]:
df = pd.read_csv("data/post_ai/adzuna/software_engineer_gb.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_software_engineer_gb.csv", index=False)


In [20]:
df = pd.read_csv("data/post_ai/adzuna/software_engineer_de.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_software_engineer_de.csv", index=False)


In [21]:
df = pd.read_csv("data/post_ai/adzuna/software_engineer_nl.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_software_engineer_nl.csv", index=False)


In [22]:
import glob

files = glob.glob("data/post_ai/adzuna/*")
files


['data/post_ai/adzuna/accountant_gb.csv',
 'data/post_ai/adzuna/cleaned_accountant_de.csv',
 'data/post_ai/adzuna/cleaned_software_engineer_nl.csv',
 'data/post_ai/adzuna/cleaned_business_analyst_nl.csv',
 'data/post_ai/adzuna/business_analyst_gb.csv',
 'data/post_ai/adzuna/cleaned_data_analyst_de.csv',
 'data/post_ai/adzuna/data_analyst_nl.csv',
 'data/post_ai/adzuna/cleaned_software_engineer_gb.csv',
 'data/post_ai/adzuna/cleaned_business_analyst_de.csv',
 'data/post_ai/adzuna/cleaned_software_engineer_de.csv',
 'data/post_ai/adzuna/cleaned_data_analyst_gb.csv',
 'data/post_ai/adzuna/accountant_nl.csv',
 'data/post_ai/adzuna/cleaned_accountant_gb.csv',
 'data/post_ai/adzuna/software_engineer_nl.csv',
 'data/post_ai/adzuna/cleaned_accountant_nl.csv',
 'data/post_ai/adzuna/cleaned_business_analyst_gb.csv',
 'data/post_ai/adzuna/data_analyst_gb.csv',
 'data/post_ai/adzuna/software_engineer_gb.csv',
 'data/post_ai/adzuna/cleaned_data_analyst_nl.csv',
 'data/post_ai/adzuna/data_analyst_de

In [23]:
import pandas as pd
import glob

cleaned_files = glob.glob("data/post_ai/adzuna/cleaned_*.csv")

for f in cleaned_files:
    print(f"\n--- Preview of {f} ---")
    df = pd.read_csv(f)
    print(df.head(5))



--- Preview of data/post_ai/adzuna/cleaned_accountant_de.csv ---
                   job_title  \
0  Senior Accountant (m/w/d)   
1  Senior Accountant (m/w/d)   
2  Senior Accountant (m/w/d)   
3  Senior Accountant (m/w/d)   
4     Tax Accountant (m/w/d)   

                                     job_description country  \
0  Trek ist ein fantastischer Arbeitgeber mit tol...      de   
1  Als Projektentwickler und Bauträger und Teil d...      de   
2  Die Michels Trenchless GmbH in Wiesbaden ist T...      de   
3  Als eines der weltweit führenden Dentalunterne...      de   
4  Sie möchten in einem international agierenden ...      de   

            date_posted  source  
0  2026-02-23T18:30:58Z  adzuna  
1  2026-02-25T06:35:04Z  adzuna  
2  2026-02-21T06:22:40Z  adzuna  
3  2026-03-02T18:44:03Z  adzuna  
4  2026-02-19T06:13:59Z  adzuna  

--- Preview of data/post_ai/adzuna/cleaned_software_engineer_nl.csv ---
                job_title                                    job_description  \

In [24]:
df = pd.read_csv("data/post_ai/adzuna/cleaned_data_analyst_gb.csv")
df.head().T


,0,1,2,3,4
job_title,Data Analyst Trainee - job guarantee,Data Analyst,Data Analyst,Data Analyst Trainee,Data Analyst
job_description,Are you ready to start a new career in Data An...,"Data Analyst Gillingham, Kent Hybrid working D...",A well-established business is looking for a D...,Are you looking to benefit from a new career i...,Data Analyst (SystmOne) Huddersfield 3 months ...
country,gb,gb,gb,gb,gb
date_posted,2026-02-19T18:28:35Z,2026-02-27T14:55:41Z,2026-02-27T22:35:38Z,2026-03-10T10:22:24Z,2026-03-12T17:32:11Z
source,adzuna,adzuna,adzuna,adzuna,adzuna


In [25]:
import pandas as pd
import glob

# find all cleaned files
cleaned_files = glob.glob("data/post_ai/adzuna/cleaned_*.csv")

# load them into a list of DataFrames
dfs = [pd.read_csv(f) for f in cleaned_files]

# merge into one dataset
adzuna_post_ai = pd.concat(dfs, ignore_index=True)

# save the merged dataset
adzuna_post_ai.to_csv("data/post_ai/adzuna/adzuna_post_ai_merged.csv", index=False)

adzuna_post_ai.head()


,job_title,job_description,country,date_posted,source
0,Senior Accountant (m/w/d),Trek ist ein fantastischer Arbeitgeber mit tol...,de,2026-02-23T18:30:58Z,adzuna
1,Senior Accountant (m/w/d),Als Projektentwickler und Bauträger und Teil d...,de,2026-02-25T06:35:04Z,adzuna
2,Senior Accountant (m/w/d),Die Michels Trenchless GmbH in Wiesbaden ist T...,de,2026-02-21T06:22:40Z,adzuna
3,Senior Accountant (m/w/d),Als eines der weltweit führenden Dentalunterne...,de,2026-03-02T18:44:03Z,adzuna
4,Tax Accountant (m/w/d),Sie möchten in einem international agierenden ...,de,2026-02-19T06:13:59Z,adzuna


In [26]:
adzuna_post_ai.shape


(600, 5)

In [27]:
import pandas as pd

# load your merged dataset
df = pd.read_csv("data/post_ai/adzuna/adzuna_post_ai_merged.csv")

def assign_role(title):
    title_lower = str(title).lower()
    if "accountant" in title_lower:
        return "accountant"
    if "business analyst" in title_lower:
        return "business_analyst"
    if "data analyst" in title_lower:
        return "data_analyst"
    if "software engineer" in title_lower or "developer" in title_lower:
        return "software_engineer"
    return "unknown"

df["role"] = df["job_title"].apply(assign_role)

df.to_csv("data/post_ai/adzuna/adzuna_post_ai_with_roles.csv", index=False)

df.head(150)


,job_title,job_description,country,date_posted,source,role
0,Senior Accountant (m/w/d),Trek ist ein fantastischer Arbeitgeber mit tol...,de,2026-02-23T18:30:58Z,adzuna,accountant
1,Senior Accountant (m/w/d),Als Projektentwickler und Bauträger und Teil d...,de,2026-02-25T06:35:04Z,adzuna,accountant
2,Senior Accountant (m/w/d),Die Michels Trenchless GmbH in Wiesbaden ist T...,de,2026-02-21T06:22:40Z,adzuna,accountant
3,Senior Accountant (m/w/d),Als eines der weltweit führenden Dentalunterne...,de,2026-03-02T18:44:03Z,adzuna,accountant
4,Tax Accountant (m/w/d),Sie möchten in einem international agierenden ...,de,2026-02-19T06:13:59Z,adzuna,accountant
...,...,...,...,...,...,...
145,Security Business Analist,Position Description: Security Business Analis...,nl,2026-03-05T02:26:41Z,adzuna,unknown
146,Technical Business Analyst,Technical Business Analyst Who we are: You may...,nl,2026-03-06T02:10:31Z,adzuna,business_analyst
147,Business Analyst Logistics,Zet jij data om in daadkracht? Kom dan werken ...,nl,2026-03-09T02:40:28Z,adzuna,business_analyst
148,Agile Business Analist,Ben je een bruggenbouwer tussen business en IT...,nl,2026-02-18T03:58:40Z,adzuna,unknown


In [28]:
df = pd.read_csv("data/post_ai/adzuna/adzuna_post_ai_with_roles.csv")
df["role"].value_counts()


,count
role,
accountant,148
software_engineer,148
business_analyst,127
data_analyst,111
unknown,66


In [29]:
df[df["role"] == "unknown"][["job_title"]].head(30)


,job_title
99,Software Test Engineer
102,Business analist SAP
106,Business Analist
107,Business analist
108,Business Analist
109,Business Analist
110,Business Analist
114,Business Analist
115,Business Analist
117,Business Analist


In [30]:
def assign_role(title):
    t = str(title).lower()

    # Accountant
    if "accountant" in t or "accounts assistant" in t or "finance assistant" in t:
        return "accountant"

    # Business Analyst
    if "business analyst" in t or "ba " in t or "business intelligence analyst" in t:
        return "business_analyst"

    # Data Analyst (English + Dutch + variants)
    if (
        "data analyst" in t or
        "data analist" in t or
        "data-analist" in t or
        "data governance" in t or
        "data operations" in t or
        "reporting analyst" in t or
        "bi analyst" in t or
        "analytics" in t
    ):
        return "data_analyst"

    # Software Engineer / Developer / Data Engineer
    if (
        "software engineer" in t or
        "developer" in t or
        "engineer" in t or
        "data engineer" in t or
        "full stack" in t or
        "backend" in t or
        "frontend" in t
    ):
        return "software_engineer"

    return "unknown"


In [31]:
df["role"] = df["job_title"].apply(assign_role)
df["role"].value_counts()


,count
role,
software_engineer,149
accountant,148
data_analyst,148
business_analyst,127
unknown,28


In [32]:
df[df["role"] == "unknown"]["job_title"].sample(20, random_state=42)


,job_title
121,Business Analist
324,Tech Lead (m/w/d) Personalization
117,Business Analist
148,Agile Business Analist
102,Business analist SAP
133,Senior Business Analist
141,Business Analist BI
183,E-Commerce Data & Insights Analyst (m/w/d)
132,Business Analist - Pancompany
135,Business Analist Assetmanagement


In [33]:
def assign_role(title):
    t = str(title).lower()

    # Accountant (English + Dutch)
    if (
        "accountant" in t or
        "boekhoudkundig" in t or
        "accounts assistant" in t or
        "finance assistant" in t
    ):
        return "accountant"

    # Business Analyst (English + Dutch + German + variants)
    if (
        "business analyst" in t or
        "business analist" in t or
        "business-analist" in t or
        "business intelligence analist" in t or
        "business intelligence analyst" in t or
        "business it-analyst" in t or
        "business systems analyst" in t or
        "business applications" in t or
        "market analyst" in t or
        "analyst prozessdigitalisierung" in t or
        "business & market analyst" in t
    ):
        return "business_analyst"

    # Data Analyst (English + Dutch + German + variants)
    if (
        "data analyst" in t or
        "data analist" in t or
        "data-analist" in t or
        "data governance" in t or
        "data operations" in t or
        "reporting analyst" in t or
        "bi analyst" in t or
        "analytics" in t
    ):
        return "data_analyst"

    # Software Engineer / Developer / Data Engineer (English + German)
    if (
        "software engineer" in t or
        "developer" in t or
        "entwickler" in t or
        "engineer" in t or
        "data engineer" in t or
        "full stack" in t or
        "backend" in t or
        "frontend" in t
    ):
        return "software_engineer"

    return "unknown"


In [34]:
df["role"] = df["job_title"].apply(assign_role)
df["role"].value_counts()


,count
role,
software_engineer,149
business_analyst,149
accountant,148
data_analyst,148
unknown,6


In [35]:
df = df[df["role"] != "unknown"]
df.to_csv("data/post_ai/adzuna/adzuna_post_ai_final.csv", index=False)


In [36]:
import pandas as pd

uk = pd.read_csv("/content/UKJobPostings.csv")

uk = uk.rename(columns={
    "title": "job_title",
    "description": "job_description",
    "date": "date_posted",
    "location": "country"
})

uk["country"] = "gb"
uk["source"] = "kaggle_uk_2023_2025"

keep = ["job_title", "job_description", "country", "date_posted", "source"]
uk = uk[[c for c in keep if c in uk.columns]]


FileNotFoundError: [Errno 2] No such file or directory: '/content/UKJobPostings.csv'

In [ ]:
import pandas as pd

uk = pd.read_csv("/content/UKJobPostings.csv")


print("UK columns:", uk.columns.tolist())



In [ ]:
import pandas as pd

uk = pd.read_csv("/content/UKJobPostings.csv")

uk_clean = uk.rename(columns={
    "title": "job_title",
    "full_description": "job_description",
    "created": "date_posted",
    "location": "country"
})

# If full_description is missing, fall back to clean_text
uk_clean["job_description"] = uk_clean["job_description"].fillna(uk["clean_text"])

uk_clean["country"] = "gb"
uk_clean["source"] = "kaggle_uk_2023_2025"

keep = ["job_title", "job_description", "country", "date_posted", "source"]
uk_clean = uk_clean[keep]


In [ ]:
uk_clean["role"] = uk_clean["job_title"].apply(assign_role)



In [ ]:
uk_clean = uk_clean[uk_clean["role"] != "unknown"]



In [ ]:
import pandas as pd
import numpy as np


In [ ]:
adzuna_clean = pd.read_csv("/content/data/post_ai/adzuna/adzuna_post_ai_final.csv")


In [ ]:
uk_clean = pd.read_csv("/content/UKJobPostings.csv")


In [ ]:
adzuna_clean.head()
uk_clean.head()


In [ ]:
post_ai_full = pd.concat([adzuna_clean, uk_clean], ignore_index=True)


In [ ]:
post_ai_full.to_csv("/content/data/post_ai/post_ai_full.csv", index=False)


In [ ]:
post_ai_full.shape
post_ai_full["role"].value_counts()
post_ai_full["country"].value_counts()


In [ ]:
adzuna_clean.shape
uk_clean.shape
post_ai_full.shape

In [ ]:
post_ai_full[post_ai_full["job_title"].str.contains("cloud", case=False, na=False)].head()


In [ ]:
post_ai_full.isna().sum()


In [ ]:
post_ai_full = post_ai_full[[
    "job_title",
    "job_description",
    "country",
    "date_posted",
    "source",
    "role"
]]
